In [11]:
import json
from pathlib import Path
from typing import Any
from langchain_mcp_adapters.client import MultiServerMCPClient


BASE_DIR = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
DEFAULT_CONFIG_PATH = BASE_DIR / "config.json"



def load_server_config(config_path: str | Path = DEFAULT_CONFIG_PATH) -> dict[str, Any]:
    """config.json 파일을 읽어 MCP 서버 설정을 로드"""

    config_path = Path(config_path)

    if not config_path.exists():
        raise FileNotFoundError(f"설정 파일을 찾을 수 없습니다: {config_path}")

    with config_path.open("r", encoding="utf-8") as f:
        config = json.load(f)

    return config



async def get_send_email_tool(config_path: str | Path = DEFAULT_CONFIG_PATH):
    """MCP 서버에 연결하여 send_email Tool을 가져옴"""

    config = load_server_config(config_path)
    client = MultiServerMCPClient(config)
    tools = await client.get_tools()

    for tool in tools:
        print(tool.name)
        print(tool.args_schema)

    send_email_tool = next(
        (tool for tool in tools if tool.name == "send_email"),
        None,
    )

    if send_email_tool is None:
        available_tools = [tool.name for tool in tools]

        raise RuntimeError(f"send_email Tool을 찾을 수 없습니다. 현재 사용 가능한 Tool 목록: {available_tools}")

    return send_email_tool

In [12]:
send_email_tool = await get_send_email_tool(DEFAULT_CONFIG_PATH)

UnsupportedOperation: fileno